# 08 · Vehicle Energy Dataset: profile before monitoring

Phase 9, Step 9.1. The second dataset: GPS and OBD-II signals of 383 (in fact 384) cars in Ann
Arbor, Nov 2017 to Nov 2018, from https://github.com/gsoh/VED (Apache License 2.0). This
notebook measures what the configuration of Steps 9.2 to 9.5 needs: how to read the files,
the columns and their ranges, which signals each engine type has, how a row gets its time,
rows per month, duplicates and the signals that should agree.

**Before running:**
```
python src/data_download/fetch_archives.py --dataset ved_signals
```
The last section prints a verdict to paste back.

In [ ]:
import json
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from profiling import ved_profile as vp

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
BLUE, INK, SURFACE = "#3987e5", "#52514e", "#fcfcfb"
SOURCE, CACHE = vp.folders()
result = vp.profile(SOURCE, CACHE)   # first run: reads every week into the Parquet cache
summary = vp.summary(result)
print(f"{summary['files']} weekly file(s), {summary['rows']:,} rows, "
      f"{summary['first_time']} to {summary['last_time']}")

## 1 · Files, and reading them without losing rows

Every file's header ends with CR LF while its rows end with LF. DuckDB's CSV reader (1.5)
then skips the first 32 MB of a file without an error, so the profile reads with pyarrow and
checks every file's row count against its line count. The cell after the table shows the loss
on the first week.

In [ ]:
files = result["files"]
print(f"rows lost by the reader: {summary['rows_lost_by_reader']}; header CR LF in "
      f"{summary['files_header_crlf']} of {summary['files']} files, rows CR LF in "
      f"{summary['files_rows_crlf']}")
files[["file", "week_start", "size_bytes", "data_lines", "rows_read", "vehicles", "trips",
       "first_time", "last_time"]]

In [ ]:
first = SOURCE / files["file"].iloc[0]
names = first.open().readline().strip().split(",")
columns = {n: "DOUBLE" for n in names}
naive = duckdb.sql(
    f"SELECT count(*) FROM read_csv('{first.as_posix()}', delim=',', header=true, quote='', "
    f"auto_detect=false, nullstr='NaN', columns={columns!r}, new_line='\\n')"
).fetchone()[0]
lines = int(files["data_lines"].iloc[0])
print(f"{first.name}: {lines:,} rows in the file, DuckDB reads {naive:,} "
      f"({lines - naive:,} lost, {(lines - naive) / lines:.1%})")

## 2 · Columns: types and ranges

In [ ]:
result["columns"].round(4)

## 3 · Which signals each engine type has

Share of NULL values per column and engine type (from the static data). Engine signals are
empty for the EVs, battery signals for petrol cars (ICE): completeness rules must depend on the
engine type.

In [ ]:
nulls = result["nulls_by_engine"].set_index("engine_type")
signals = nulls.drop(columns=["rows", "DayNum", "VehId", "Trip", "Timestamp(ms)",
                              "Latitude[deg]", "Longitude[deg]"])
fig = go.Figure(go.Heatmap(
    z=signals.values, x=signals.columns, y=signals.index, zmin=0, zmax=1,
    colorscale=[[0, "#f3f7fd"], [1, "#7aaaea"]], colorbar=dict(title="NULL share"),
    textfont=dict(color=INK),
    text=signals.map(lambda v: f"{v:.0%}").values, texttemplate="%{text}",
    hovertemplate="%{y}, %{x}: %{z:.1%} NULL<extra></extra>",
))
fig.update_layout(title="Missing values by engine type", height=380, plot_bgcolor=SURFACE,
                  xaxis_tickangle=-40, margin=dict(b=170),
                  yaxis_autorange="reversed")
fig.show()
nulls[["rows"]].T

## 4 · Vehicles, trips, duplicates

In [ ]:
print(json.dumps({k: result[k] for k in ("vehicles", "trips", "duplicates")}, indent=2,
                 default=str))
print("static data, columns per file:", json.dumps(result["static_columns"], indent=2))
result["static_quality"]

## 5 · Time: from DayNum and Timestamp(ms) to a row time

DayNum is the trip's start in days (1.0 = 2017-11-01 00:00), the same for every row of a trip;
Timestamp(ms) is the time since the trip started. A row's time is their sum. Rows are grouped
into months by that time; trips that cross midnight at a week's end stay in their start's
file.

In [ ]:
months = result["months"]
fig = go.Figure(go.Bar(x=months["month"], y=months["rows"], marker_color=BLUE,
                       hovertemplate="%{x}: %{y:,} rows<extra></extra>"))
fig.update_layout(title="Rows per month (by row time)", yaxis_title="rows", plot_bgcolor=SURFACE,
                  bargap=0.25)
fig.show()
print(result["outside_week"])
months

## 6 · GPS area and signals that should agree

In [ ]:
print(json.dumps(result["gps"], indent=2))
result["consistency"]

## 7 · Verdict\n\nCopy the printed text back.

In [ ]:
verdict = {
    **{k: summary[k] for k in ("files", "rows", "rows_lost_by_reader", "first_time", "last_time")},
    "duckdb_naive_loss_first_week": lines - naive,
    "vehicles_by_engine_type": result["vehicles"]["vehicles_with_rows_by_engine_type"],
    "trips": result["trips"]["trips"],
    "duplicates_on_key": result["duplicates"]["extra_rows_on_key"],
    "rows_after_week_end": result["outside_week"]["rows_after_week_end"],
    "months": summary["months"],
    "negative_ms_rows": int(duckdb.sql(
        f"SELECT count(*) FROM read_parquet('{(CACHE / '*.parquet').as_posix()}') "
        "WHERE \"Timestamp(ms)\" < 0").fetchone()[0]),
}
print(json.dumps(verdict, indent=2, default=str))